In [ ]:
%load_ext autoreload
%autoreload 2
    
import polars as pl
from polars import col
import plotly.express as px
from jsr.utils.processing import process_dataframes
from jsr.utils.col_namespace import Sched, Ship, Trans
from plotly.subplots import make_subplots
from jsr.plotting.plot_transactions import *

In [ ]:
dataframes = process_dataframes("../data")
df_schedule = dataframes["schedule"]
df_shipments = dataframes["shipments"]
df_transactions = dataframes["transactions"]

## Timelines

In [ ]:
k = (df_transactions
 # .filter(col(Trans.START_TIME).dt.day() == 21)
     # .filter(col(Trans.START_TIME).dt.day() == 10)
 .filter(col(Trans.START_TIME).dt.day() == 14)
 .filter(col(Trans.START_TIME).dt.month() == 11)
 # .filter(col(Trans.OPERATOR_ID).is_in([245, ]))
 .sort("OPERATOR_ID")
 .with_columns(col(Trans.OPERATOR_ID).cast(str))
 # .filter(col(Trans.OPERATOR_ID) == 35)
)#.sort("START_TIME")

fig = px.timeline(k, x_start="START_TIME",
                  # x_end="END_TIME", y="OPERATOR_ID", color=Trans.TRANSACTION_TYPE)
                  x_end="END_TIME", y="OPERATOR_ID", color=Trans.FEAT_SHIFT_ID)
fig.update_traces(width=0.98)
fig.update_layout(bargap=0.01,bargroupgap=0.0)
fig.update_yaxes(tickfont_size=4)
fig.add_vline(x=datetime.datetime(2020, 11, 14, 19), line_dash="dash", line_color="black", line_width=2)
fig.add_vline(x=datetime.datetime(2020, 11, 14, 7), line_dash="dash", line_color="black", line_width=2)

In [ ]:
ship_demo1_transaction_schedule(df_transactions, write_image=True)

In [ ]:
ship_demo2_transaction_schedule(df_transactions, write_image=True)

In [ ]:
ship_demo_outlier(df_transactions, write_image=True)

## Shift Stats

## Getting Outlier Stats

In [ ]:
zero_dur = len(df_transactions.filter(col(Trans.FEAT_DURATION) == 0))
zero_qnt = len(df_transactions.filter(col(Trans.TRANSACTION_QTY) == 0))
long_dur = len(df_transactions.filter(col(Trans.FEAT_DURATION) > 21600)) # 6 hours
total_transactions = len(df_transactions)
print("{} out of {} ({:.1f}%) transactions are 0 duration".format(
    zero_dur, total_transactions, 100 * zero_dur / total_transactions))
print("{} out of {} ({:.1f}%) transactions are 0 quantity".format(
    zero_qnt, total_transactions, 100 * zero_qnt / total_transactions))
print("{} out of {} transactions are longer than 6 hours".format(long_dur, total_transactions))



In [ ]:
q01 = col(Trans.FEAT_DURATION).quantile(0.01).alias("q01")
q99 = col(Trans.FEAT_DURATION).quantile(0.99).alias("q99")
q05 = col(Trans.FEAT_DURATION).quantile(0.01).alias("q05")
q95 = col(Trans.FEAT_DURATION).quantile(0.99).alias("q95")
outlier_quantiles = (
    df_transactions
    .select(q05, q95)
)
outlier_quantiles

In [ ]:
type_outliers = (
    df_transactions
    .group_by(Trans.TRANSACTION_TYPE)
    .agg(q05, q95)
)

In [ ]:
type_outliers

In [ ]:
df_filtered_transactions = (
    df_transactions
    .join(type_outliers, on=Trans.TRANSACTION_TYPE, how="left")
    .filter(col(Trans.FEAT_DURATION) > col("q05"))
    .filter(col(Trans.FEAT_DURATION) < col("q95"))
    .drop("q05", "q95")
)